<a href="https://colab.research.google.com/github/dklishta/python-ai-Gailunaite-Darya/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- [`cartoons_genre_country_duration.csv`](https://github.com/componavt/python-ai-template/blob/main/data/examples/cartoons_genre_country_duration.sparql) — жанры, страны и продолжительность мультфильмов
- [`cartoons_assessment_reviews.csv`](https://github.com/componavt/python-ai-template/blob/main/data/examples/cartoons_assessment_reviews.sparql) — оценки и рецензии мультфильмов

**Что мы делаем:**
1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файлы в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [ ]:
# 🐱 Шаг 1. Клонируем ваш репозиторий курса в Colab

import os

repo = "python-ai-Gailunaite-Darya"  # ← ИЗМЕНЕНО: имя вашего репозитория
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):          # всегда проверяет /content/python-ai-Gailunaite-Darya
    !git clone -q https://github.com/dklishta/python-ai-Gailunaite-Darya.git  # ← ИЗМЕНЕНО: URL вашего репозитория

if os.getcwd() != repo_path:               # точное сравнение, не endswith
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Gailunaite-Darya


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [ ]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

import pandas as pd

df_teacoffee = pd.read_csv("data/tea_coffee_info.csv")
df_teacoffeefin = pd.read_csv("data/tea_coffee_fin.csv")

print("✅ Загружено строк в df_teacoffee:", len(df_teacoffee))
print("✅ Загружено строк в df_teacoffeefin:", len(df_teacoffeefin))

✅ Загружено строк в df_teacoffee: 289
✅ Загружено строк в df_teacoffeefin: 1555


## 🧹 [2B] Очистка и переименование столбцов

В вашем CSV-файле `tea_coffee_info.csv` есть **технические столбцы**, которые требуют обработки:

- Столбец `company` с URL (ссылкой на объект Wikidata) — **удаляем**, так как для анализа он не нужен.
- Столбцы с суффиксом `Label` (`companyLabel`, `productTypeLabel`, `countryLabel`, `hqLabel`) содержат читаемые названия — **переименуем их**, убрав постфикс `Label`.
- Столбец `hqCoord` содержит координаты в формате WKT — **оставляем как есть** (для возможной визуализации на карте).
- Столбец `foundingYearShort` содержит год основания — **приведём к числовому типу** для фильтрации и сортировки.

В этом шаге мы:
- **удаляем** столбец `company` (URL Wikidata);
- **переименовываем**: `companyLabel → company`, `productTypeLabel → productType`, `countryLabel → country`, `hqLabel → hq`;
- **приводим** `foundingYearShort` к типу `int` (целое число).

При приведении к числам мы используем:
- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные → `NaN`;
- `fillna(0)` — заменяет пропуски на 0;
- `astype(int)` — переводит столбец к целочисленному типу.

> ⚠️ **Важно:** после этого шага у вас останется аккуратная таблица с понятными названиями столбцов, готовая к фильтрации, группировке и визуализации.

In [12]:
# 🧹 Шаг 2B. Очистка и переименование столбцов для tea_coffee_info.csv
# (с защитой от NameError и улучшенной загрузкой данных)

import pandas as pd
import os

# 🔁 Если df не определён — пробуем загрузить файл заново
if 'df' not in globals():
    print("⚠️ Переменная 'df' не найдена. Пробуем загрузить данные...")

    file_path = "data/tea_coffee_info.csv"

    # Проверяем существование файла
    if not os.path.exists(file_path):
        # Пробуем альтернативные пути (частая проблема в Colab)
        alt_paths = [
            "/content/python-ai-Gailunaite-Darya/data/tea_coffee_info.csv",
            "tea_coffee_info.csv",
            "../data/tea_coffee_info.csv"
        ]
        for alt in alt_paths:
            if os.path.exists(alt):
                file_path = alt
                print(f"✅ Файл найден по альтернативному пути: {file_path}")
                break
        else:
            raise FileNotFoundError(
                f"❌ Файл не найден!\n"
                f"Текущая папка: {os.getcwd()}\n"
                f"Ожидаемый путь: data/tea_coffee_info.csv\n"
                f"Содержимое текущей папки: {os.listdir('.')}"
            )

    # Пробуем разные кодировки и разделители
    df = None
    for encoding in ["utf-8", "cp1251", "utf-8-sig"]:
        for sep in [",", ";", "\t"]:
            try:
                df = pd.read_csv(file_path, encoding=encoding, sep=sep)
                print(f"✅ Файл успешно загружен: encoding='{encoding}', sep='{sep}'")
                break
            except Exception:
                continue
        if df is not None:
            break

    if df is None:
        raise ValueError(
            "❌ Не удалось прочитать CSV-файл.\n"
            "Проверьте: кодировку файла, разделитель, целостность данных."
        )

# ============================================================
# 🧹 Основная логика очистки и переименования
# ============================================================

# 1) Удаляем технический столбец с URL Wikidata (если существует)
if "company" in df.columns:
    df = df.drop(columns=["company"])
    print("✅ Столбец 'company' (URL) удалён")
else:
    print("⏭️ Столбец 'company' не найден, пропускаем удаление")

# 2) Переименовываем столбцы: убираем суффикс Label
rename_map = {
    "companyLabel": "company",
    "productTypeLabel": "productType",
    "countryLabel": "country",
    "hqLabel": "hq",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
print("✅ Столбцы переименованы:", list(df.columns))

# 3) Приводим foundingYearShort к числовому типу (если столбец существует)
if "foundingYearShort" in df.columns:
    df["foundingYearShort"] = pd.to_numeric(
        df["foundingYearShort"], errors="coerce"
    ).fillna(0).astype(int)
    print("✅ foundingYearShort приведён к типу int")
else:
    print("⏭️ Столбец foundingYearShort не найден, пропускаем преобразование")

# 4) Показываем результат
print("\n📋 Первые 5 строк после очистки:")
display(df.head())

print(f"\n📊 Информация о DataFrame: {df.shape[0]} строк, {df.shape[1]} столбцов")
print("\n✅ Данные готовы к анализу")

⏭️ Столбец 'company' не найден, пропускаем удаление
✅ Столбцы переименованы: ['productType', 'country', 'hq', 'hqCoord', 'foundingYearShort']
✅ foundingYearShort приведён к типу int

📋 Первые 5 строк после очистки:


,productType,country,hq,hqCoord,foundingYearShort
0,чай,Соединённое королевство Великобритании и Ирландии,Лондон,Point(-0.1275 51.507222222),1600
1,чай,Соединённое королевство Великобритании и Ирландии,Лондон,Point(-0.1275 51.507222222),1600
2,кофе,Канада,Оквилл,Point(-79.683333333 43.45),1964
3,кофе,Швейцария,Лозанна,Point(6.633333333 46.533333333),1986
4,кофе,США,Нортфилд,Point(-87.766666666 42.1),2012



📊 Информация о DataFrame: 289 строк, 5 столбцов

✅ Данные готовы к анализу


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame с данными о чайных и кофейных компаниях:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по **году основания** (`foundingYearShort`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы выводить информацию в структурированном виде.

**Ваши столбцы после очистки:**
- `company` — название компании
- `productType` — тип продукта (чай, кофе и т.д.)
- `country` — страна происхождения
- `hq` — город штаб-квартиры
- `hqCoord` — координаты штаб-квартиры (WKT-формат)
- `foundingYearShort` — год основания (число)

In [13]:
# 🔍 Шаг 3. Обзор данных

def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("=" * 60)
    print("Размер:", df.shape, f"({df.shape[0]} строк, {df.shape[1]} столбцов)")
    print("Столбцы:", ", ".join(df.columns))
    print("\n📋 Первые строки:")
    display(df.head(n))

    # 📈 Базовая статистика по числовым столбцам
    numeric_cols = df.select_dtypes(include=['int', 'float']).columns
    if len(numeric_cols) > 0:
        print("\n📈 Статистика по числовым столбцам:")
        print(df[numeric_cols].describe())
    else:
        print("\n⚠️ Нет числовых столбцов для статистики")

    # 🏷️ Уникальные значения категориальных столбцов
    print("\n🏷️ Уникальные значения:")
    for col in df.select_dtypes(include=['object']).columns:
        unique_count = df[col].nunique()
        print(f"  • {col}: {unique_count} уникальных значений")

# ============================================================
# 🔍 Вызов функции для вашего DataFrame
# ============================================================

show_info(df, "Чайные и кофейные компании (df)")

# ============================================================
# 💡 Дополнительно: быстрая сводка по вашим данным
# ============================================================

print("\n" + "=" * 60)
print("📌 БЫСТРАЯ СВОДКА ПО ДАННЫМ")
print("=" * 60)

if "productType" in df.columns:
    print("\n☕ Типы продуктов:")
    print(df["productType"].value_counts())

if "country" in df.columns:
    print("\n🌍 Страны:")
    print(df["country"].value_counts())

if "foundingYearShort" in df.columns:
    print("\n📅 Год основания:")
    print(f"  • Самый ранний: {df['foundingYearShort'].min()}")
    print(f"  • Самый поздний: {df['foundingYearShort'].max()}")
    print(f"  • Средний: {df['foundingYearShort'].mean():.0f}")

print("\n✅ Обзор данных завершён")


📊 Чайные и кофейные компании (df)
Размер: (289, 5) (289 строк, 5 столбцов)
Столбцы: productType, country, hq, hqCoord, foundingYearShort

📋 Первые строки:


,productType,country,hq,hqCoord,foundingYearShort
0,чай,Соединённое королевство Великобритании и Ирландии,Лондон,Point(-0.1275 51.507222222),1600
1,чай,Соединённое королевство Великобритании и Ирландии,Лондон,Point(-0.1275 51.507222222),1600
2,кофе,Канада,Оквилл,Point(-79.683333333 43.45),1964
3,кофе,Швейцария,Лозанна,Point(6.633333333 46.533333333),1986
4,кофе,США,Нортфилд,Point(-87.766666666 42.1),2012



📈 Статистика по числовым столбцам:
       foundingYearShort
count         289.000000
mean         1612.733564
std           732.986245
min             0.000000
25%          1845.000000
50%          1933.000000
75%          1993.000000
max          2022.000000

🏷️ Уникальные значения:
  • productType: 2 уникальных значений
  • country: 50 уникальных значений
  • hq: 145 уникальных значений
  • hqCoord: 150 уникальных значений

📌 БЫСТРАЯ СВОДКА ПО ДАННЫМ

☕ Типы продуктов:
productType
кофе    178
чай     111
Name: count, dtype: int64

🌍 Страны:
country
США                                                  41
Нидерланды                                           24
Великобритания                                       18
Канада                                               11
Соединённое королевство Великобритании и Ирландии    10
Австралия                                             9
Индия                                                 7
Дания                                             

## 🌍 [4.3] Географический рейтинг: какие страны лидируют?

В этом блоке мы отвечаем на вопрос: **из каких стран родом большинство компаний**, представленных в нашем датасете?

### 🔎 Что мы анализируем:
- **Топ-10 стран** по количеству записей о компаниях;
- **Долю каждой страны** в процентах от общего числа записей;
- **Разбивку по типу продукта** (чай / кофе) внутри топ-стран;
- **Накопленную долю** (cumulative share) — сколько процентов данных покрывает топ-N стран.

### 📊 Почему это важно:
- Показывает исторические центры чайной и кофейной торговли;
- Помогает выявить географические «слепые зоны» (регионы, слабо представленные в Викиданных);
- Даёт основу для дальнейшей фильтрации: например, «показать только европейские компании».

### 🧮 Методы pandas, которые мы используем:
| Метод | Зачем нужен |
|-------|-------------|
| `value_counts()` | Считает частоту каждого значения в столбце |
| `normalize=True` | Возвращает доли вместо абсолютных чисел |
| `cumsum()` | Считает накопленную сумму (для cumulative share) |
| `groupby() + size()` | Группирует данные по нескольким столбцам |
| `unstack()` | Превращает группировку в сводную таблицу |

> 💡 **Подсказка**: если в столбце `country` есть пропуски (`NaN`), они автоматически исключаются из `value_counts()`. Мы явно проверим это и сообщим о пропущенных значениях.

In [16]:
# 🌍 Шаг 4.3. Географический рейтинг: какие страны лидируют?

import pandas as pd

print("🌍 АНАЛИЗ: Географическое распределение компаний")
print("=" * 70)

# ============================================================
# 🔍 0. Предварительная проверка данных
# ============================================================

if "country" not in df.columns:
    raise ValueError("❌ Столбец 'country' не найден в DataFrame!\n"
                     f"Доступные столбцы: {list(df.columns)}")

# Проверяем пропуски в столбце country
missing_countries = df["country"].isna().sum()
total_rows = len(df)
print(f"📋 Всего записей: {total_rows}")
print(f"⚠️  Записей без указания страны: {missing_countries} "
      f"({missing_countries/total_rows*100:.1f}%)")

# Фильтруем данные: оставляем только строки с указанной страной
df_countries = df[df["country"].notna()].copy()
print(f"✅ Записей для анализа: {len(df_countries)}\n")

# ============================================================
# 📊 1. Топ-10 стран по количеству компаний
# ============================================================

print("🏆 ТОП-10 СТРАН ПО КОЛИЧЕСТВУ КОМПАНИЙ")
print("-" * 70)

country_counts = df_countries["country"].value_counts()
country_pct = df_countries["country"].value_counts(normalize=True) * 100
country_cumulative = country_pct.cumsum()

# Создаём сводную таблицу для вывода
top_n = 10
summary = pd.DataFrame({
    "Страна": country_counts.index[:top_n],
    "Количество": country_counts.values[:top_n],
    "Доля, %": country_pct.values[:top_n],
    "Накопленная доля, %": country_cumulative.values[:top_n]
})

# Форматируем вывод
for _, row in summary.iterrows():
    bar = "█" * int(row["Доля, %"] / 2)  # мини-гистограмма текстом
    print(f"{row['Страна']:<40} {row['Количество']:>3} | {row['Доля, %']:>5.1f}% {bar}")

print(f"\n📈 Топ-{top_n} стран покрывают {country_cumulative.iloc[min(top_n-1, len(country_cumulative)-1)]:.1f}% всех записей")

# ============================================================
# ☕🆚🫖 2. Разбивка по типу продукта внутри топ-5 стран
# ============================================================

if "productType" in df_countries.columns:
    print("\n\n☕🆚🫖 РАЗБИВКА ПО ПРОДУКТАМ В ТОП-5 СТРАНАХ")
    print("-" * 70)

    top_5_countries = country_counts.index[:5]
    df_top5 = df_countries[df_countries["country"].isin(top_5_countries)]

    # Сводная таблица: страна × продукт
    product_pivot = df_top5.groupby(["country", "productType"]).size().unstack(fill_value=0)

    # Добавляем итоговый столбец
    product_pivot["Всего"] = product_pivot.sum(axis=1)

    # Сортируем по убыванию и выводим
    product_pivot = product_pivot.sort_values("Всего", ascending=False)

    print(product_pivot.to_markdown(index=True, tablefmt="grid"))

# ============================================================
# 🧮 3. Дополнительные метрики
# ============================================================

print("\n\n📊 ДОПОЛНИТЕЛЬНЫЕ МЕТРИКИ")
print("-" * 70)

print(f"• Всего уникальных стран: {df_countries['country'].nunique()}")
print(f"• Средняя частота страны: {len(df_countries) / df_countries['country'].nunique():.2f} компаний/страна")
print(f"• Медианная частота: {df_countries['country'].value_counts().median():.0f} компаний")

# Страны с одной компанией (long tail)
single_country_count = (country_counts == 1).sum()
print(f"• Стран с только одной компанией: {single_country_count} "
      f"({single_country_count / len(country_counts) * 100:.1f}% от всех стран)")

# ============================================================
# 💾 4. (Опционально) Сохранение результата
# ============================================================

# Если хотите сохранить рейтинг для отчёта:
# country_report = pd.DataFrame({
#     "country": country_counts.index,
#     "count": country_counts.values,
#     "percent": country_pct.values
# }).round(2)
# country_report.to_csv("data/country_ranking.csv", index=False, encoding="utf-8-sig")
# print("\n✅ Рейтинг стран сохранён в data/country_ranking.csv")

print("\n✅ Географический анализ завершён")

🌍 АНАЛИЗ: Географическое распределение компаний
📋 Всего записей: 289
⚠️  Записей без указания страны: 70 (24.2%)
✅ Записей для анализа: 219

🏆 ТОП-10 СТРАН ПО КОЛИЧЕСТВУ КОМПАНИЙ
----------------------------------------------------------------------
США                                       41 |  18.7% █████████
Нидерланды                                24 |  11.0% █████
Великобритания                            18 |   8.2% ████
Канада                                    11 |   5.0% ██
Соединённое королевство Великобритании и Ирландии  10 |   4.6% ██
Австралия                                  9 |   4.1% ██
Индия                                      7 |   3.2% █
Дания                                      7 |   3.2% █
Германия                                   7 |   3.2% █
Франция                                    6 |   2.7% █

📈 Топ-10 стран покрывают 63.9% всех записей


☕🆚🫖 РАЗБИВКА ПО ПРОДУКТАМ В ТОП-5 СТРАНАХ
----------------------------------------------------------------------
+--

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей
  - типы оценок и результатов

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨